# 가벼운 포즈 생성 개선 실험

기존에 실행한 Stable Diffusion 1.5 + OpenPose ControlNet으로 돌아가 B 포즈 한 장을 비교합니다. 512×512 전신 이미지를 생성하며 얼굴 수정보다는 팔과 자세를 확인합니다. SDXL이나 인페인팅 모델은 사용하지 않습니다.

이 노트는 로컬에서 문법만 확인했고 Colab 실행은 아직 확인하지 않았습니다. 무료 GPU가 보장되거나 결과가 반드시 좋아지는 것은 아닙니다. 이전 SDXL 세션이 RAM 부족으로 종료됐으므로 새 세션에서 이 노트만 실행합니다.

두 결과는 포즈 조건 적용이 끝나는 시점만 다릅니다. 기준 1.0은 마지막까지, 실험 0.8은 전체 생성의 약 80%까지 포즈 조건을 적용합니다. 포즈와 자연스러움이 각각 어떻게 바뀌는지 비교하며, 다른 조건은 동일하게 유지합니다.

모델: [SD 1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), [OpenPose ControlNet](https://huggingface.co/lllyasviel/control_v11p_sd15_openpose). 공개 전에 모델 이용 조건과 참조 사진 권리를 확인하세요.

## 1. GPU 확인과 설치
런타임 유형에서 GPU를 선택합니다. 설치 완료 후 런타임 재시작을 요구하면 재시작하고 설치 셀은 생략한 채 GPU 확인과 다음 셀을 실행하세요. Gradio는 사용하지 않습니다.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형에서 GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 이전에 설치 완료 메시지를 확인한 버전 조합입니다. Gradio 충돌 경고가 생길 수 있습니다.
%pip install 'diffusers==0.39.0' 'transformers==4.57.6' accelerate 'controlnet-aux==0.0.10' 'huggingface-hub==0.36.2' safetensors

## 2. B 포즈 사진 한 장 업로드
두 팔이 잘 보이는 사진을 골라 주세요. 공개 가능한 사진만 사용합니다. 파일 선택 창에서 한 장만 선택합니다.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display
uploaded = files.upload()
names = list(uploaded)
if len(names) != 1:
    raise ValueError('사진을 한 장 선택하고 셀을 다시 실행하세요.')
out = Path('/content/pose_light')
out.mkdir(parents=True, exist_ok=True)
source = ImageOps.exif_transpose(Image.open(names[0])).convert('RGB')
reference = ImageOps.pad(source, (512, 512), method=Image.Resampling.LANCZOS, color='white')
reference.save(out / 'reference.png')
print('선택한 사진:', names[0])
display(reference)

## 3. 포즈 추출
원본과 뼈대를 비교해 팔·다리의 위치가 맞는지 먼저 확인합니다. B 사진은 팔이 몸 앞을 지나므로 뼈대가 정확해도 잘못 생성될 수 있습니다.

In [ ]:
from controlnet_aux import OpenposeDetector
import gc
detector = OpenposeDetector.from_pretrained('lllyasviel/ControlNet')
skeleton = detector(reference).resize((512, 512))
skeleton.save(out / 'skeleton.png')
display(reference, skeleton)
del detector
gc.collect()
torch.cuda.empty_cache()

## 4. 기존 모델 로드
이전에 이미지 생성에 사용한 모델과 같은 계열을 사용합니다. 처음에는 모델 다운로드가 필요합니다. CPU 오프로딩은 GPU 메모리 절약에 도움이 되지만 시스템 RAM을 사용합니다.

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler
base_model = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
control_model = 'lllyasviel/control_v11p_sd15_openpose'
controlnet = ControlNetModel.from_pretrained(control_model, torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(base_model, controlnet=controlnet, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
print('모델 준비 완료, pipe:', 'pipe' in globals())

## 5. 같은 조건으로 두 장 비교
기준과 실험의 차이는 `control_guidance_end`뿐입니다. 원래 사용한 문장과 seed 42, steps 30, control strength 0.8을 유지합니다. 두 장을 한 장씩 차례로 생성합니다. 기존 실험과도 비교할 수 있지만 실행 환경 차이가 있을 수 있습니다.

In [ ]:
import json, torch
from importlib.metadata import version
prompt = 'a single dancer in casual clothes in a bright dance studio, full body, photograph'
seed, steps, strength = 42, 30, 0.8
records = []
for label, end in [('baseline_end1', 1.0), ('experiment_end08', 0.8)]:
    print('생성 중:', label)
    generated = pipe(prompt=prompt, image=skeleton, width=512, height=512,
        generator=torch.Generator(device='cpu').manual_seed(seed),
        num_inference_steps=steps, guidance_scale=7.5,
        controlnet_conditioning_scale=strength, control_guidance_end=end).images[0]
    generated.save(out / (label + '.png'))
    display(generated)
    records.append({'name':label, 'prompt':prompt, 'seed':seed, 'steps':steps,
        'control_strength':strength, 'control_guidance_end':end, 'guidance_scale':7.5,
        'base_model':base_model, 'control_model':control_model, 'size':[512,512],
        'gpu':torch.cuda.get_device_name(0), 'diffusers':version('diffusers'),
        'input_file':names[0]})
    (out / 'settings.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print('저장 완료:', label)

## 6. 결과 다운로드
원본·뼈대·생성된 결과·설정이 ZIP에 포함됩니다. 공개 전 이미지의 권리와 개인정보를 확인하세요.

In [ ]:
import shutil
archive = shutil.make_archive('/content/pose_light_results', 'zip', out)
files.download(archive)

## 실험 관찰 — 실행 후 작성
- 기준 결과에서 팔 개수와 B 포즈 반영: 미확인
- 80% 적용 결과에서 달라진 점: 미확인
- 자연스러움과 포즈 정확도 중 좋아지거나 나빠진 점: 미확인
- GPU·실행 시간·오류: 미확인